[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C08_Training_Systems_Course/04_flashattention/04_flashattention.ipynb)

# 04 · FlashAttention：Online Softmax 与 IO 复杂度

<span style="background:#1f6feb;color:#fff;padding:2px 8px;border-radius:4px;font-size:12px">CPU</span>
从零手写 online softmax 与分块 attention，验证它逐位等于标准实现却不存 S×S 矩阵，并算 IO 复杂度差异。

**你将完成：**
1. 数值稳定的标准 softmax / attention（baseline）
2. **online softmax**：流式 running max/sum，逐位等于标准
3. **flash-style 分块 attention**：输出与标准一致，但显存 O(S) 不是 O(S²)
4. IO 复杂度账：为什么"算得更多反而更快"

> 用真实 pythia-1.4b 的 head 维度 d=128 作实验设置。

## 0 · 设置（用真实模型的 head 维度）

In [ ]:
import os, json, urllib.request
import numpy as np
CACHE=os.path.expanduser("~/.training_systems_data"); os.makedirs(CACHE, exist_ok=True)
MODELS={"pythia-1.4b":"https://huggingface.co/EleutherAI/pythia-1.4b/resolve/main/config.json"}
def load_config(m):
    p=os.path.join(CACHE,f"{m}.json")
    if not os.path.exists(p): urllib.request.urlretrieve(MODELS[m],p)
    c=json.load(open(p)); g=lambda *k: next(c[x] for x in k if x in c); h=g("hidden_size","n_embd")
    return dict(L=g("num_hidden_layers","n_layer"),h=h,heads=g("num_attention_heads","n_head"))
cfg=load_config("pythia-1.4b")
d = cfg["h"]//cfg["heads"]    # 每个 head 的维度
print(f"pythia-1.4b: hidden={cfg['h']}, heads={cfg['heads']} -> head_dim d={d}")
rng=np.random.default_rng(0)
S=256
Q=rng.normal(size=(S,d)).astype(np.float64)
K=rng.normal(size=(S,d)).astype(np.float64)
V=rng.normal(size=(S,d)).astype(np.float64)
print(f"序列长度 S={S}, head_dim d={d}")

## 1 · 标准 attention（baseline）

显式算 S×S 分数矩阵，softmax，乘 V。这是要被超越的参照。

In [ ]:
def softmax_rows(Z):
    Z = Z - Z.max(1, keepdims=True)
    e = np.exp(Z); return e/e.sum(1, keepdims=True)

def attention_standard(Q,K,V):
    scores = Q@K.T / np.sqrt(Q.shape[1])    # (S,S) <- 显式物化！
    return softmax_rows(scores) @ V

O_ref = attention_standard(Q,K,V)
print("标准 attention 输出 shape:", O_ref.shape)
print(f"它物化了 {S}x{S} = {S*S} 个分数 -> O(S²) 显存")

## 2 · Online softmax：流式 running max/sum

不存整行，分块扫描，每块更新 (m, ℓ) 并用校正因子折算。验证逐位等于标准 softmax。

In [ ]:
def online_softmax(scores_row, block=32):
    "对一行分数做 online softmax，返回归一化权重。"
    n=len(scores_row); m=-np.inf; l=0.0
    # 第一遍流式求 m,l
    for s in range(0,n,block):
        blk=scores_row[s:s+block]; m_new=max(m, blk.max())
        l = np.exp(m-m_new)*l + np.exp(blk-m_new).sum()
        m=m_new
    return np.exp(scores_row-m)/l

row = (Q@K.T/np.sqrt(d))[0]
w_online = online_softmax(row)
w_std = softmax_rows(row[None])[0]
print("online vs 标准 softmax 最大误差:", np.abs(w_online-w_std).max())
print("逐位相等:", np.allclose(w_online, w_std), "✓ —— 流式算法不丢精度")

## 3 · Flash-style 分块 attention

外层 Q 块、内层 K/V 块，online 累积输出 O 与统计量 (m, ℓ)，从不物化整个 S×S。
验证输出与标准 attention 一致。

In [ ]:
def attention_flash(Q,K,V,block=32):
    S,d=Q.shape; scale=1/np.sqrt(d)
    O=np.zeros((S,d));
    for qs in range(0,S,block):
        Qb=Q[qs:qs+block]; bm=Qb.shape[0]
        m=np.full(bm,-np.inf); l=np.zeros(bm); acc=np.zeros((bm,d))
        for ks in range(0,S,block):
            Kb=K[ks:ks+block]; Vb=V[ks:ks+block]
            S_blk=Qb@Kb.T*scale                      # (bm, kb) 只在"片上"
            m_new=np.maximum(m, S_blk.max(1))
            corr=np.exp(m-m_new)                      # 校正因子
            p=np.exp(S_blk-m_new[:,None])
            l=corr*l + p.sum(1)
            acc=corr[:,None]*acc + p@Vb
            m=m_new
        O[qs:qs+block]=acc/l[:,None]
    return O

O_flash=attention_flash(Q,K,V)
print("flash vs 标准 attention 最大误差:", np.abs(O_flash-O_ref).max())
print("逐位相等:", np.allclose(O_flash,O_ref), "✓ —— 不物化 S² 矩阵，结果完全一样")

## 4 · IO 复杂度：为什么更快

标准 attention HBM 访问 O(S²)；flash O(S²d/M)。算长序列下的差异。

In [ ]:
def hbm_standard(S,d): return S*S + S*d            # 物化分数矩阵 + 读写
def hbm_flash(S,d,M=100000): return (S*S*d)/M + S*d   # 分块，M=SRAM 大小
for S_ in [1024, 8192, 32768]:
    std=hbm_standard(S_,d); fl=hbm_flash(S_,d)
    print(f"S={S_:6d}: 标准 HBM≈{std:.2e}  flash≈{fl:.2e}  减少 {std/fl:.1f}x")
print("\n=> S 越长差距越大；attention 是 memory-bound，省 HBM 往返直接省时间")
print("   注意 flash 的 FLOPs 并不更少（甚至略多），快在 IO 而非算力")

## 5 · 注意力为什么是 memory-bound（算术强度账）

FlashAttention 优化 IO 而非 FLOPs，前提是 attention 本就 memory-bound。这里把 $QK^\top$ 这一步的算术强度（FLOPs/byte）算出来，和 A100 的拐点强度（312T / 2TB·s ≈ 156）比一比——验证它确实落在拐点之下，于是减少 HBM 往返才是对症的优化。

In [ ]:
# 为什么 attention 值得做成 IO 优化？因为它本就 memory-bound。
# 算 QK^T 这一步的算术强度，和 A100 拐点(312T/2TB·s≈156)比。
def attn_score_intensity(S, d, dtype_bytes=2):
    flops = 2*S*S*d                                  # QK^T: S×S×d 的乘加
    bytes_ = (S*d + S*d + S*S)*dtype_bytes           # 读 Q,K + 写 S×S 分数
    return flops/bytes_
ridge = 312e12/2e12                                  # A100 fp16 拐点强度
print(f"A100 拐点强度 = 312T/2TB·s ≈ {ridge:.0f} FLOP/byte")
print(f"{'S':>7s} {'QK^T 算术强度':>16s} {'判定':>14s}")
for S_ in [512, 2048, 8192]:
    ai=attn_score_intensity(S_, d)
    print(f"{S_:7d} {ai:15.1f} {'compute' if ai>ridge else 'memory-bound':>14s}")
# 自检：典型 seq 下 attention 远低于拐点 -> memory-bound -> 优化 IO 才对症
assert attn_score_intensity(2048, d) < ridge, "attention 应 memory-bound"
print("\n=> attention 的算术强度远在拐点之下，是 memory-bound；")
print("   所以 FlashAttention 不去优化 FLOPs，而是减少 HBM 往返(上一节的 IO 账)")

---
## ✏️ 练习区

### ✏️ 练习 1：数值稳定 softmax

实现 `stable_softmax(z)`（1D 向量，先减 max）。验证大 logit 不溢出。

In [ ]:
def stable_softmax(z):
    # TODO: 减 max 再 exp 归一化
    raise NotImplementedError


In [ ]:
# —— 练习 1 自测 ——
assert np.allclose(stable_softmax(np.array([1000.,1001.,999.])).sum(), 1.0)
assert np.isfinite(stable_softmax(np.array([1e4,1e4]))).all()
assert np.allclose(stable_softmax(np.zeros(4)), 0.25)
print("练习 1 通过 ✓")


### ✏️ 练习 2：online softmax 等于标准

实现 `online_softmax_weights(row, block)`：分块流式，返回归一化权重，逐位等于标准 softmax。

In [ ]:
def online_softmax_weights(row, block=16):
    # TODO: 维护 running m, l；遇更大 max 用 exp(m_old-m_new) 校正旧 l
    raise NotImplementedError


In [ ]:
# —— 练习 2 自测 ——
rng2=np.random.default_rng(3)
for _ in range(5):
    row=rng2.normal(0,5,size=100)
    assert np.allclose(online_softmax_weights(row,16), stable_softmax(row), atol=1e-12)
# 不同 block 大小结果一致
r=rng2.normal(size=64)
assert np.allclose(online_softmax_weights(r,8), online_softmax_weights(r,32))
print("练习 2 通过 ✓  online softmax 与标准逐位相等，且与块大小无关")


### ✏️ 练习 3：flash 分块 attention

实现 `flash_attention(Q,K,V,block)`，输出与标准 attention 一致。

In [ ]:
def flash_attention(Q,K,V,block=32):
    # TODO: 外层 Q 块、内层 K/V 块；维护 m,l,acc；用校正因子；最后 acc/l
    raise NotImplementedError


In [ ]:
# —— 练习 3 自测 ——
O_mine=flash_attention(Q,K,V,block=64)
assert np.allclose(O_mine, O_ref, atol=1e-10), "flash 输出应等于标准 attention"
# 块大小不影响结果
assert np.allclose(flash_attention(Q,K,V,16), flash_attention(Q,K,V,128), atol=1e-10)
print("练习 3 通过 ✓  你实现了 FlashAttention 的数学内核")


### ✏️ 练习 4：IO 复杂度账

实现 `io_reduction(S, d, M)`：返回标准 attention 与 flash 的 HBM 访问比值（标准/flash）。
验证 S 越大比值越大。

In [ ]:
def io_reduction(S, d, M=100000):
    # TODO: hbm_std = S*S + S*d ; hbm_flash = S*S*d/M + S*d ; 返回比值
    raise NotImplementedError


In [ ]:
# —— 练习 4 自测 ——
assert io_reduction(8192, d) > io_reduction(1024, d), "长序列下 flash 优势更大"
assert io_reduction(32768, d) > 1, "flash 应减少 IO"
print(f"练习 4 通过 ✓  S=1k减少{io_reduction(1024,d):.1f}x, S=32k减少{io_reduction(32768,d):.1f}x")


---
## 📖 参考答案

In [ ]:
# 练习 1
def stable_softmax(z):
    z=z-z.max(); e=np.exp(z); return e/e.sum()
print("练习 1 ✓")

In [ ]:
# 练习 2
def online_softmax_weights(row, block=16):
    n=len(row); m=-np.inf; l=0.0
    for s in range(0,n,block):
        blk=row[s:s+block]; m_new=max(m, blk.max())
        l=np.exp(m-m_new)*l + np.exp(blk-m_new).sum(); m=m_new
    return np.exp(row-m)/l
print("练习 2 ✓")

In [ ]:
# 练习 3
def flash_attention(Q,K,V,block=32):
    S,d=Q.shape; scale=1/np.sqrt(d); O=np.zeros((S,d))
    for qs in range(0,S,block):
        Qb=Q[qs:qs+block]; bm=Qb.shape[0]
        m=np.full(bm,-np.inf); l=np.zeros(bm); acc=np.zeros((bm,d))
        for ks in range(0,S,block):
            Sb=Qb@K[ks:ks+block].T*scale
            m_new=np.maximum(m,Sb.max(1)); corr=np.exp(m-m_new)
            p=np.exp(Sb-m_new[:,None]); l=corr*l+p.sum(1)
            acc=corr[:,None]*acc + p@V[ks:ks+block]; m=m_new
        O[qs:qs+block]=acc/l[:,None]
    return O
print("练习 3 ✓")

In [ ]:
# 练习 4
def io_reduction(S, d, M=100000):
    return (S*S + S*d) / (S*S*d/M + S*d)
print("练习 4 ✓ —— 优化对瓶颈(IO)，不是优化看起来大的那个数(FLOPs)")